# Vector Data Base creation

This notebook reads `.csv` files in `chunks/` folder, then encodes every chunk into a vector using
each of the three models (BERT, HateBERT, RoBERTa) × three weight variants (IHC fine-tuned, ISHate fine-tuned, base),
and saves three FAISS indices per model-variant setup.

**Outputs** (27 FAISS indices + 3 shared lookup tables):
```
index/
  bert/
    IHC/    vdb_training.faiss  vdb_documents.faiss  vdb_full.faiss
    ISHate/ vdb_training.faiss  vdb_documents.faiss  vdb_full.faiss
    base/   vdb_training.faiss  vdb_documents.faiss  vdb_full.faiss
  hatebert/   (same structure)
  roberta/    (same structure)
  lookup_training.json
  lookup_documents.json
  lookup_full.json
```

**Key design choices:**
- Embedding: `[CLS]` token, `last_hidden_state[:, 0, :]`
- Similarity: cosine, via L2-normalization + `IndexFlatIP` (exact search)
- Index type: `IndexIDMap(IndexFlatIP)`, FAISS returns `chunk_id` directly
- Document chunk IDs are offset by `100_000` to avoid collision with training IDs in `vdb_full`

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Configurations

Here we define all the different configs, some hyperparameters and paths

In [3]:
RAG_DIR     = Path(".")
WEIGHTS_DIR = Path("..") / "weights"
INDEX_DIR   = RAG_DIR / "index"

ALL_MODELS = {
    "bert": {
        "hf_id": "bert-base-uncased",
        "variants": {
            "IHC":        {"weights_path": str(WEIGHTS_DIR / "bert-base-uncased_IHC"),        "finetuned": True},
            "ISHate":     {"weights_path": str(WEIGHTS_DIR / "bert-base-uncased_ISHate"),     "finetuned": True},
            "Vicomtech":  {"weights_path": str(WEIGHTS_DIR / "bert-base-uncased_Vicomtech"),  "finetuned": True},
            "base":       {"weights_path": None,                                              "finetuned": False},
        },
    },
    "hatebert": {
        "hf_id": "GroNLP/hateBERT",
        "variants": {
            "IHC":        {"weights_path": str(WEIGHTS_DIR / "hateBERT_IHC"),        "finetuned": True},
            "ISHate":     {"weights_path": str(WEIGHTS_DIR / "hateBERT_ISHate"),     "finetuned": True},
            "Vicomtech":  {"weights_path": str(WEIGHTS_DIR / "hateBERT_Vicomtech"),  "finetuned": True},
            "base":       {"weights_path": None,                                     "finetuned": False},
        },
    },
    "roberta": {
        "hf_id": "roberta-base",
        "variants": {
            "IHC":        {"weights_path": str(WEIGHTS_DIR / "roberta-base_IHC"),        "finetuned": True},
            "ISHate":     {"weights_path": str(WEIGHTS_DIR / "roberta-base_ISHate"),     "finetuned": True},
            "Vicomtech":  {"weights_path": str(WEIGHTS_DIR / "roberta-base_Vicomtech"),  "finetuned": True},
            "base":       {"weights_path": None,                                          "finetuned": False},
        },
    },
}

BATCH_SIZE = 64
MAX_LENGTH = 64

## 3. Load Chunks

Load both source .csv files to vectorize:
- `chunks_training.csv`
- `chunks_documents.csv`

Three source tuples are prepared for the indexing loop: `training`, `documents` and `full` taht combine both.

In [4]:
DOCUMENTS_ID_OFFSET = 100_000  # training IDs: 0-99999 and document IDs: 100000+ to provide overlaps

df_training  = pd.read_csv("chunks/chunks_training.csv")[["chunk_id", "text"]]
df_documents = pd.read_csv("chunks/chunks_documents.csv")[["chunk_id", "text"]]
df_full = pd.concat([
    df_training,
    df_documents.assign(chunk_id=df_documents["chunk_id"] + DOCUMENTS_ID_OFFSET),
], ignore_index=True)

SOURCES = {
    "training":  (df_training["text"].tolist(),  df_training["chunk_id"].to_numpy(dtype="int64")),
    "documents": (df_documents["text"].tolist(), df_documents["chunk_id"].to_numpy(dtype="int64")),
    "full":      (df_full["text"].tolist(),       df_full["chunk_id"].to_numpy(dtype="int64")),
}

print(f"Training chunks : {len(df_training):,}")
print(f"Document chunks : {len(df_documents):,}")
print(f"Full (combined) : {len(df_full):,}")

Training chunks : 67,864
Document chunks : 40,950
Full (combined) : 108,814


## 4. Encoding Function

Takes a list of strings + a loaded model + tokenizer, returns a float32 numpy array of shape
`(N, 768)` where each row is the `[CLS]` embedding of one chunk.

Uses batched inference with `torch.no_grad()`. Moves tensors to GPU if available.

In [5]:
# This function is now in the rag.py file
from rag import encode

## 5. Build FAISS Indices

For each model × variant × source:
1. Load tokenizer from HuggingFace; load encoder weights (fine-tuned via `.base_model`, or base `AutoModel`)
2. Encode all chunks → `(N, 768)` float32 array
3. L2-normalize for cosine similarity, build `IndexIDMap(IndexFlatIP)`
4. Save to `index/{model}/{variant}/vdb_{source}.faiss`

Tokenizer is shared across variants of the same model (loaded once per model).

## a. Inspect Saved Weights

Before encoding, verify two things per model:
1. **Key prefix** — what do the keys in `model.safetensors` actually look like? A mismatch with `AutoModel` would silently leave the model at base HuggingFace weights.
2. **Classifier head** — are classification-head keys (`classifier.*`, `pooler.*`) present in the saved file? They should NOT end up in the final `AutoModel` (confirmed by `unexpected` keys being dropped via `strict=False`), but we want to see them explicitly.

In [6]:
from safetensors.torch import load_file as load_safetensors

for model_name, model_cfg in ALL_MODELS.items():
    for variant_name, var_cfg in model_cfg["variants"].items():
        if not var_cfg["finetuned"]:
            continue
        safetensors_path = os.path.join(var_cfg["weights_path"], "model.safetensors")
        if not os.path.isfile(safetensors_path):
            print(f"[{model_name}/{variant_name}] No model.safetensors — skipping")
            continue

        saved_keys = list(load_safetensors(safetensors_path).keys())
        head_keys  = [k for k in saved_keys if any(p in k for p in ("classifier", "pooler"))]
        full_model = AutoModelForSequenceClassification.from_pretrained(var_cfg["weights_path"])
        matched    = sum(1 for k in saved_keys if k in full_model.state_dict())

        print(f"\n{'='*50}")
        print(f"{model_name} / {variant_name}")
        print(f"  Keys matched    : {matched} / {len(saved_keys)}")
        print(f"  Head keys       : {head_keys}")
        print(f"  base_model type : {type(full_model.base_model).__name__}")
        del full_model


bert / IHC
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



bert / ISHate
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



bert / Vicomtech
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



hatebert / IHC
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



hatebert / ISHate
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



hatebert / Vicomtech
  Keys matched    : 201 / 201
  Head keys       : ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
  base_model type : BertModel



roberta / IHC
  Keys matched    : 201 / 201
  Head keys       : ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
  base_model type : RobertaModel



roberta / ISHate
  Keys matched    : 201 / 201
  Head keys       : ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
  base_model type : RobertaModel



roberta / Vicomtech
  Keys matched    : 201 / 201
  Head keys       : ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
  base_model type : RobertaModel


## b. Encode and Build Indices

For each model × variant × source, encode all chunks and save the FAISS index to `index/{model}/{variant}/vdb_{source}.faiss`.

In [7]:
os.makedirs(str(INDEX_DIR), exist_ok=True)

for model_name, model_cfg in ALL_MODELS.items():
    hf_id = model_cfg["hf_id"]
    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    for variant_name, var_cfg in model_cfg["variants"].items():
        variant_dir = INDEX_DIR / model_name / variant_name
        variant_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n=== {model_name} / {variant_name} ===")

        if var_cfg["finetuned"]:
            full_model = AutoModelForSequenceClassification.from_pretrained(var_cfg["weights_path"])
            model = full_model.base_model
        else:
            model = AutoModel.from_pretrained(hf_id)
        model.eval().to(device)

        embed_dim = model.config.hidden_size

        for source_name, (texts, chunk_ids) in SOURCES.items():
            vectors = encode(texts, model, tokenizer)
            assert vectors.shape == (len(texts), embed_dim), \
                f"Unexpected shape {vectors.shape}, expected ({len(texts)}, {embed_dim})"
            faiss.normalize_L2(vectors)

            inner = faiss.IndexFlatIP(embed_dim)
            index = faiss.IndexIDMap(inner)
            index.add_with_ids(vectors, chunk_ids)

            out_path = variant_dir / f"vdb_{source_name}.faiss"
            faiss.write_index(index, str(out_path))
            print(f"  [{source_name}] {index.ntotal} vectors → {out_path.relative_to(RAG_DIR)}")

        del model
        if var_cfg["finetuned"]:
            del full_model
        if device.type == "cuda":
            torch.cuda.empty_cache()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


=== bert / IHC ===


...

Encoding: 100%|█████████▉| 1695/1701 [01:15<00:00, 20.68it/s]

Encoding: 100%|█████████▉| 1698/1701 [01:15<00:00, 20.48it/s]

Encoding: 100%|██████████| 1701/1701 [01:15<00:00, 22.14it/s]

Encoding: 100%|██████████| 1701/1701 [01:15<00:00, 22.45it/s]

  [full] 108814 vectors → index/roberta/base/vdb_full.faiss


## 6. Save Shared Lookup Tables

Three JSON files at the top level of `index/` map `chunk_id → text`.
These are shared across all model/variant combinations — the text is identical regardless of how it was encoded.

In [8]:
def _save_lookup(path, ids, texts):
    d = {str(cid): text for cid, text in zip(ids, texts)}
    with open(path, "w") as f:
        json.dump(d, f, ensure_ascii=False, indent=2)
    print(f"Saved {Path(path).relative_to(RAG_DIR)}  ({len(d):,} entries)")

_save_lookup(INDEX_DIR / "lookup_training.json",  df_training["chunk_id"],  df_training["text"])
_save_lookup(INDEX_DIR / "lookup_documents.json", df_documents["chunk_id"], df_documents["text"])
_save_lookup(INDEX_DIR / "lookup_full.json",      df_full["chunk_id"],      df_full["text"])

Saved index/lookup_training.json  (67,864 entries)


Saved index/lookup_documents.json  (40,950 entries)


Saved index/lookup_full.json  (108,814 entries)


## 7. Sanity Check

Load the hateBERT index and run two sample queries to verify the pipeline end-to-end.

> If result counts vary wildly across queries (e.g. sometimes 0, sometimes 10), `SIMILARITY_THRESHOLD` is too strict. A dedicated test notebook will measure this and switch to top-k if needed.

In [9]:
SIMILARITY_THRESHOLD = 0.95

_sc_model   = "roberta"
_sc_variant = "ISHate"
_sc_source  = "training"

_sc_cfg  = ALL_MODELS[_sc_model]
_sc_var  = _sc_cfg["variants"][_sc_variant]

test_index = faiss.read_index(str(INDEX_DIR / _sc_model / _sc_variant / f"vdb_{_sc_source}.faiss"))
with open(INDEX_DIR / f"lookup_{_sc_source}.json") as f:
    lookup = json.load(f)

tokenizer_sc = AutoTokenizer.from_pretrained(_sc_cfg["hf_id"])
full_sc      = AutoModelForSequenceClassification.from_pretrained(_sc_var["weights_path"])
model_sc     = full_sc.base_model.eval().to(device)

def query_index(tweet_text, k=5, threshold=SIMILARITY_THRESHOLD):
    vec = encode([tweet_text], model_sc, tokenizer_sc, batch_size=1)
    faiss.normalize_L2(vec)
    scores, ids = test_index.search(vec, k)
    return [
        (lookup[str(ids[0][i])], float(scores[0][i]))
        for i in range(k)
        if ids[0][i] != -1 and scores[0][i] >= threshold
    ]

queries = [
    ("HATE",     "jews are now in full control : canadian man goes to jail for posting on white nationalist websites"),
    ("HATE",     "i dont like neggers"),
    ("NOT HATE", "can they breath fire and have scaly skin ? sure they can light a cross on fire but that doesn't count ."),
    ("NOT HATE", "Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman 's fiancé , mom ."),
]

print(f"Sanity check — {_sc_model}/{_sc_variant} on {_sc_source}\n")
for label, tweet in queries:
    print(f"[{label}] {tweet}")
    hits = query_index(tweet)
    if hits:
        for text, score in hits:
            print(f"  {score:.3f}  {text}")
    else:
        print("  (no results above threshold)")
    print()

Sanity check — roberta/ISHate on training

[HATE] jews are now in full control : canadian man goes to jail for posting on white nationalist websites


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 141.08it/s]

  0.998  [hate] so dan carlin is uninformed about the is narrative to america first that nordics testament to the anti white coming
  0.998  [hate] . on #periscope : the cuckturians assaulting white nationalists : the latest
  0.998  [hate]  ah  the joys of diversity :  " : south africas police arresting young women to rape them ... via " " "
  0.998  [hate] white genocide accelerates in sweden : university removes all portaits of white male professors #swrm
  0.998  [hate]   " diverse " " britain : black gangs to be rewarded with free barbecues and sports events | more : "

[HATE] i dont like neggers


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

...

Encoding: 100%|██████████| 1/1 [00:00<00:00, 139.38it/s]

  0.997  [not hate] can they breath fire  fly  and have scaly skin ? sure they can light a cross on fire but that doesn't count .
  0.997  [hate] debate comments news jews created fake jewish graves this is an opener especially
  0.997  [hate] catch they mutilate their own and they do their best to see to it that those whom they see as their goyim take hold of slaves are also marked
  0.997  [hate] `` there is no god but allah , and religion is a religion of peace . yes , where peace stands for silence for the victims of terrorist attacks . ''
  0.997  [hate] `` there is no god but allah , and islam is a faith of peace . yes , where peace stands for silence for the victims of terrorist attacks . ''

[NOT HATE] Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman 's fiancé , mom .


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 157.04it/s]

  0.990  [not hate] Read more : Charges in deadly wreck that killed 5 motorcyclists - LancasterOnline.com News Crash takes injured woman 's fiancé , mom - LancasterOnline.com News
  0.989  [not hate] Jackson 's retrial took jurors back to the events of late summer 1977 , when two widows , Vernita Curtis , 81 and Gladys Ott , 90 , were beaten and strangled to death during burglaries a week apart in a Long Beach apartment building .
  0.988  [not hate] YouTube - Grandson attacks and rapes 87-Year-Old bed-ridden amputee Grandmother YouTube - Police are looking for Flash Mob of black Teens who Robbed Dupont Circle Store .
  0.987  [not hate] 32-year-old Jason `` Jay '' Tate was found with Kelsey Sue Stahl 's car .
  0.987  [not hate] Utah nonprofit helps hundreds of refugees go to college  Up until she was 13, the only place Kai Sin called home was the refugee camp where she was born after her family escaped ethnic violence in Myanmar.  https://t.co/0vpNLPni6D

